# GPT-500M Inference Notebook

**Architecture:** Decoder-only GPT with multi-head causal self-attention  
**Scale:** ~505M parameters (28 layers × 1152 embd × 16 heads, GPT-2 BPE vocab)  
**Purpose:** Load a saved checkpoint and run autoregressive text generation  

**Flow:** Mount Drive → Install deps → Define model → Load checkpoint → Generate

---
Edit **Section 6 – Generation Config** to change the prompt, temperature, and sampling settings.

## 1. Install Dependencies

In [ ]:
# Install dependencies (run once per Colab session)
!pip install -q tiktoken

## 2. Imports & Device Setup

In [ ]:
import os
import gc
import glob
import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
if device == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"PyTorch : {torch.__version__}")

Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB
PyTorch : 2.11.0+cu128


## 3. Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths (edit to match your training setup) ─────────────────────────────────
CKPT_DIR = '/content/drive/MyDrive/Colab Notebooks/GPT500M/Checkpoints'

print(f"Checkpoint dir : {CKPT_DIR}")

checkpoints = sorted(
    glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")),
    key=os.path.getmtime
)
if checkpoints:
    print(f"\nFound {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        size_mb = os.path.getsize(ckpt) / 1e6
        print(f"  {os.path.basename(ckpt)}  ({size_mb:.0f} MB)")
    print(f"\nWill load: {os.path.basename(checkpoints[-1])}  (latest)")
else:
    print("No checkpoints found in CKPT_DIR — check the path above.")

Mounted at /content/drive
Checkpoint dir : /content/drive/MyDrive/Colab Notebooks/GPT500M/Checkpoints

Found 1 checkpoint(s):
  ckpt_001000.pt  (6062 MB)

Will load: ckpt_001000.pt  (latest)


## 4. Model Architecture

Identical to the training notebook — must match exactly for `load_state_dict` to work.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size : int   = 50257
    block_size : int   = 1024
    n_layer    : int   = 28
    n_head     : int   = 16
    n_embd     : int   = 1152
    dropout    : float = 0.0
    # Training fields kept for checkpoint compatibility; unused during inference
    num_epochs        : int   = 2
    micro_batch_size  : int   = 1
    grad_accum_steps  : int   = 128
    lr                : float = 3e-4
    weight_decay      : float = 0.1
    grad_clip         : float = 1.0
    warmup_steps      : int   = 1000
    train_steps       : int   = 112000
    eval_interval     : int   = 100
    eval_batches      : int   = 50
    tokenizer_name    : str   = "gpt2"
    max_checkpoints   : int   = 3
    device            : str   = "cuda" if torch.cuda.is_available() else "cpu"


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head   = cfg.n_head
        self.n_embd   = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout  = cfg.dropout
        self.c_attn     = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj     = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self._cache_k: Optional[torch.Tensor] = None
        self._cache_v: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = split_heads(q), split_heads(k), split_heads(v)
        if use_cache:
            if self._cache_k is not None:
                k = torch.cat([self._cache_k, k], dim=2)
                v = torch.cat([self._cache_v, v], dim=2)
            self._cache_k = k
            self._cache_v = v
        dropout_p = self.dropout if self.training else 0.0
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=dropout_p, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

    def clear_cache(self):
        self._cache_k = None
        self._cache_v = None


class FeedForward(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),
            nn.Dropout(cfg.dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd)
        self.ffn  = FeedForward(cfg)

    def _block_fn(self, x):
        x = x + self.attn(self.ln1(x), use_cache=False)
        x = x + self.ffn(self.ln2(x))
        return x

    def forward(self, x, use_cache=False):
        if self.training and not use_cache:
            return grad_ckpt.checkpoint(self._block_fn, x, use_reentrant=False)
        x = x + self.attn(self.ln1(x), use_cache=use_cache)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT500M(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg      = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb  = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop     = nn.Dropout(cfg.dropout)
        self.blocks   = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_final = nn.LayerNorm(cfg.n_embd)
        self.head     = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self._init_weights()
        n = self.num_params()
        print(f"GPT-500M | {n:,} params ({n/1e6:.1f}M) | device: {cfg.device}")

    def _init_weights(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                std = 0.02
                if name.endswith(("c_proj", "net.2")):
                    std = 0.02 / math.sqrt(2 * self.cfg.n_layer)
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def num_params(self):
        return sum(p.numel() for n, p in self.named_parameters() if not (n == "head.weight"))

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"Sequence length {T} exceeds block_size {self.cfg.block_size}"
        positions = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x      = self.ln_final(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def clear_kv_cache(self):
        for block in self.blocks:
            block.attn.clear_cache()

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=256, temperature=0.8, top_k=50, use_cache=True):
        """Autoregressive generation with top-k sampling and optional KV-cache."""
        self.eval()
        self.clear_kv_cache()
        context   = prompt
        generated = []
        for step in range(max_new_tokens):
            ctx = context[:, -self.cfg.block_size:]
            if use_cache and step > 0:
                ctx = context[:, -1:]
            logits, _ = self.forward(ctx, use_cache=use_cache)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None and top_k > 0:
                topk_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < topk_vals[:, [-1]]] = float("-inf")
            probs      = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            context    = torch.cat([context, next_token], dim=1)
            generated.append(next_token.item())
        self.clear_kv_cache()
        return torch.tensor(generated)

print("Model classes defined.")

Model classes defined.


## 5. Load Checkpoint

Uses the same two-phase CPU-first load strategy as the training notebook to avoid VRAM spikes on T4.

In [ ]:
# ── Optional: select a specific checkpoint ────────────────────────────────────
# By default the latest checkpoint is loaded. To load a specific one, set:
#   CHECKPOINT_PATH = '/content/drive/MyDrive/Colab Notebooks/GPT500M/Checkpoints/ckpt_001000.pt'
# Otherwise leave as None to auto-select the latest.
CHECKPOINT_PATH = None   # <- edit here if needed
# ─────────────────────────────────────────────────────────────────────────────


def load_model_for_inference(ckpt_path, device):
    """
    Load only the model weights from a training checkpoint.
    Returns (model, cfg, step, epoch, losses).

    Two-phase load to avoid a VRAM spike on T4:
      1. Deserialise checkpoint to CPU RAM (zero VRAM cost).
      2. Allocate model skeleton on GPU, copy weights in-place.
    """
    print(f"Loading: {ckpt_path}")
    ckpt       = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg        = ckpt["cfg"]
    cfg.device = device
    model = GPT500M(cfg).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    step   = ckpt.get("step",   0)
    epoch  = ckpt.get("epoch",  0)
    losses = ckpt.get("losses", {})
    del ckpt
    gc.collect()
    if device == "cuda":
        alloc  = torch.cuda.memory_allocated() / 1e9
        reserv = torch.cuda.memory_reserved()  / 1e9
        print(f"  VRAM : {alloc:.2f} GB allocated / {reserv:.2f} GB reserved")
    print(
        f"  Checkpoint step {step} | epoch {epoch}\n"
        + (f"  train loss {losses['train']:.4f} | val loss {losses['val']:.4f}"
           if losses else "  (no loss info in checkpoint)")
    )
    return model, cfg, step, epoch, losses


if CHECKPOINT_PATH is None:
    checkpoints = sorted(
        glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")),
        key=os.path.getmtime
    )
    assert checkpoints, "No checkpoints found — check CKPT_DIR in Section 3."
    CHECKPOINT_PATH = checkpoints[-1]
    print(f"Auto-selected latest checkpoint: {os.path.basename(CHECKPOINT_PATH)}")

# Clear any previously loaded model
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except NameError:
    pass

model, cfg, ckpt_step, ckpt_epoch, ckpt_losses = \
    load_model_for_inference(CHECKPOINT_PATH, device)
print("\nModel ready for inference.")

Auto-selected latest checkpoint: ckpt_001000.pt
Loading: /content/drive/MyDrive/Colab Notebooks/GPT500M/Checkpoints/ckpt_001000.pt
GPT-500M | 505,113,984 params (505.1M) | device: cuda
  VRAM : 2.02 GB allocated / 2.04 GB reserved
  Checkpoint step 1000 | epoch 0
  train loss 4.5851 | val loss 4.4653

Model ready for inference.


## 6. Generation Config

Edit the settings below, then run **Section 7** to generate.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

# ── Edit these ────────────────────────────────────────────────────────────────
PROMPT_TEXT    = "The derivative of sin(x) with respect to x is"
MAX_NEW_TOKENS = 256     # number of tokens to generate
TEMPERATURE    = 0.8     # higher = more random; lower = more focused
TOP_K          = 50      # keep only top-K logits (0 to disable)
USE_CACHE      = True    # KV-cache: faster inference, same output
# ─────────────────────────────────────────────────────────────────────────────

prompt_ids = enc.encode_ordinary(PROMPT_TEXT)
print(f"Prompt  : {repr(PROMPT_TEXT)}")
print(f"Tokens  : {len(prompt_ids)} → {prompt_ids}")
print(f"Settings: max_new_tokens={MAX_NEW_TOKENS}, temperature={TEMPERATURE}, top_k={TOP_K}, use_cache={USE_CACHE}")

Prompt  : 'The derivative of sin(x) with respect to x is'
Tokens  : 12 → [464, 27255, 286, 7813, 7, 87, 8, 351, 2461, 284, 2124, 318]
Settings: max_new_tokens=256, temperature=0.8, top_k=50, use_cache=True


## 7. Generate

In [ ]:
import time

prompt_tensor = torch.tensor([prompt_ids], dtype=torch.int64).to(device)

t0 = time.time()
generated_ids = model.generate(
    prompt         = prompt_tensor,
    max_new_tokens = MAX_NEW_TOKENS,
    temperature    = TEMPERATURE,
    top_k          = TOP_K,
    use_cache      = USE_CACHE,
)
elapsed = time.time() - t0

full_ids  = prompt_ids + generated_ids.tolist()
full_text = enc.decode(full_ids)

tok_per_sec = MAX_NEW_TOKENS / elapsed
print("-" * 65)
print(f"[step {ckpt_step} | epoch {ckpt_epoch} | {MAX_NEW_TOKENS} tokens in {elapsed:.1f}s ({tok_per_sec:.1f} tok/s)]")
print("-" * 65)
print(full_text)

-----------------------------------------------------------------
[step 1000 | epoch 0 | 256 tokens in 5.1s (50.7 tok/s)]
-----------------------------------------------------------------
The derivative of sin(x) with respect to x is the the function from the second-end as a $$#: the non1s,-1 of the the one-z the of a the way of the the.r of the a single line data of the the the to the a new $$ that point and the two way of the $$f the the as a the the the. matrix of the \ of the line of the as two the is the the the time: $. ( as the the matrix of the first the one of the the the number of the the the the data,- have are, of a-in to which of the first, the the way of the a have between the a the the the the the the same with another a and the the the work and the the the a.n place of an and linear the the linear approach to the the of the the two the the the as the the new 1 and of the the the the a it by that: -r’ the a to a value the $ to the value of a number of it in the the:.-al,

## 8. Batch Generation (optional)

Run multiple prompts in one go and compare outputs side by side.

In [ ]:
# ── Edit the prompts list below ───────────────────────────────────────────────
PROMPTS = [
    "The integral of x^2 is",
    "Prove that the square root of 2 is irrational.",
    "In mathematics, a prime number is",
]

BATCH_MAX_NEW_TOKENS = 128
BATCH_TEMPERATURE    = 0.7
BATCH_TOP_K          = 40
# ─────────────────────────────────────────────────────────────────────────────

for i, prompt_text in enumerate(PROMPTS):
    ids           = enc.encode_ordinary(prompt_text)
    prompt_tensor = torch.tensor([ids], dtype=torch.int64).to(device)
    gen_ids  = model.generate(
        prompt         = prompt_tensor,
        max_new_tokens = BATCH_MAX_NEW_TOKENS,
        temperature    = BATCH_TEMPERATURE,
        top_k          = BATCH_TOP_K,
        use_cache      = True,
    )
    full_ids  = ids + gen_ids.tolist()
    full_text = enc.decode(full_ids)
    print(f"\n{'=' * 65}")
    print(f"[Prompt {i+1}/{len(PROMPTS)}] {repr(prompt_text)}")
    print("-" * 65)
    print(full_text)

print(f"\n{'=' * 65}")
print("Batch generation complete.")